# 08 Evidence: intervals, annotator ceiling, granularity, thresholds
Run this notebook **on your own computer** (VS Code, conda env `aiep`), not Colab. The annotator ceiling needs the raw OPP-115 annotation files in `data/opp115/annotations/`, which are not in the GitHub repo. The BERT sentence test downloads `dralsarrani/opp115-bert` from Hugging Face (needs internet; CPU is fine).

Rule used everywhere below: **a macro-F1 difference smaller than the interval width is not evidence of anything.**

In [2]:
!pip -q install transformers datasets accelerate
!git clone -q https://github.com/dralsarrani/PDPLComplianceCopilot repo
%cd repo

/content/repo


In [3]:
import os, sys, re, json, warnings
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

import joblib, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support as prf
from src.classifier.data import CATEGORIES as CATS, clean_text

df = pd.read_csv("data/opp115_segments.csv")
df["text"] = df["text"].map(clean_text)
split = json.load(open("configs/split.json"))
te = df[df.policy_file.isin(split["test"])].reset_index(drop=True)
y = te[CATS].values

base = joblib.load("artifacts/classifier/baseline.joblib")
p_base = base["clf"].predict_proba(base["vec"].transform(te.text))
p_bert = np.load("results/test_probs_bert_v2.npy")      # BERT v2 test probabilities (saved earlier)

for name, p in [("baseline", p_base), ("BERT v2", p_bert)]:
    print(name, "test macro-F1 =", prf(y, (p > 0.5).astype(int), zero_division=0)[2].mean().round(3))

baseline test macro-F1 = 0.742
BERT v2 test macro-F1 = 0.783


## Part 1a. Bootstrap by policy (1000 resamples of the 18 test policies)
We resample whole policies, never single paragraphs, because paragraphs of one policy share wording. For each policy we count true positives, false positives and false negatives per category once. Each resample just adds up the counts of the chosen policies.

In [4]:
pols = sorted(te.policy_file.unique())
rng = np.random.default_rng(42)
idx = rng.integers(0, len(pols), size=(1000, len(pols)))        # 1000 draws of 18 policies, with replacement

def counts(p):
    pred = (p > 0.5).astype(int)
    out = []
    for pol in pols:
        m = (te.policy_file == pol).values
        tp = (pred[m] * y[m]).sum(0); fp = (pred[m] * (1 - y[m])).sum(0); fn = ((1 - pred[m]) * y[m]).sum(0)
        out.append([tp, fp, fn])
    return np.array(out)                                         # (policies, 3, classes)

f1 = {}
for name, p in [("baseline", p_base), ("BERT v2", p_bert)]:
    c = counts(p)[idx].sum(1)                                    # (1000, 3, classes)
    tp, fp, fn = c[:, 0], c[:, 1], c[:, 2]
    f1[name] = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0)   # (1000, classes)

rows = []
for name in f1:
    m = f1[name].mean(1)
    rows.append([name, m.mean(), *np.percentile(m, [2.5, 97.5])])
d = (f1["BERT v2"] - f1["baseline"]).mean(1)
rows.append(["BERT minus baseline", d.mean(), *np.percentile(d, [2.5, 97.5])])
print(pd.DataFrame(rows, columns=["macro-F1", "mean", "95% low", "95% high"]).round(3).to_string(index=False))
print("share of resamples where BERT beats baseline:", (d > 0).mean())

           macro-F1  mean  95% low  95% high
           baseline 0.739    0.685     0.775
            BERT v2 0.779    0.721     0.827
BERT minus baseline 0.040    0.004     0.082
share of resamples where BERT beats baseline: 0.983


### Per-category intervals and support
Categories with few test positives get very wide intervals. Do not read anything into their differences.

In [11]:
tab = pd.DataFrame({"support": y.sum(0)}, index=CATS)
for name in f1:
    lo, hi = np.percentile(f1[name], [2.5, 97.5], axis=0)
    tab[name + " F1"] = f1[name].mean(0)
    tab[name + " 95% CI"] = [f"{a:.2f} to {b:.2f}" for a, b in zip(lo, hi)]
dd = f1["BERT v2"] - f1["baseline"]
tab["BERT-baseline CI"] = [f"{a:+.2f} to {b:+.2f}" for a, b in zip(*np.percentile(dd, [2.5, 97.5], axis=0))]
tab.round(3)

,support,baseline F1,baseline 95% CI,BERT v2 F1,BERT v2 95% CI,BERT-baseline CI
First Party Collection/Use,196,0.811,0.78 to 0.84,0.864,0.83 to 0.89,+0.01 to +0.09
Third Party Sharing/Collection,144,0.732,0.66 to 0.79,0.830,0.76 to 0.88,+0.05 to +0.15
User Choice/Control,62,0.693,0.62 to 0.76,0.685,0.56 to 0.81,-0.10 to +0.08
"User Access, Edit and Deletion",25,0.741,0.64 to 0.84,0.810,0.67 to 0.92,-0.09 to +0.20
Data Retention,10,0.439,0.00 to 0.73,0.570,0.29 to 0.77,-0.17 to +0.46
Data Security,37,0.801,0.68 to 0.92,0.891,0.83 to 0.96,-0.00 to +0.19
Policy Change,23,0.888,0.77 to 0.98,0.775,0.67 to 0.88,-0.22 to -0.01
Do Not Track,6,0.780,0.40 to 1.00,0.780,0.40 to 1.00,+0.00 to +0.00
International and Specific Audiences,61,0.835,0.76 to 0.92,0.865,0.78 to 0.93,-0.07 to +0.11
Other,212,0.667,0.63 to 0.71,0.718,0.67 to 0.77,+0.02 to +0.09


## Part 1b. Annotation ceiling
OPP-115 has 3 annotators per policy. We trained on their majority vote, but here we score one annotator against another with the same macro-F1. That number is roughly the best any model can reach on these labels. We also score BERT against each single annotator, which is the fair comparison.

In [12]:
from src.classifier.data import load_annotations
ann = load_annotations("data/opp115")
ann = ann[ann.policy_file.isin(split["test"])]
cidx = {c: i for i, c in enumerate(CATS)}
row = {(p, s): i for i, (p, s) in enumerate(zip(te.policy_file, te.segment_id))}

mats = {}                                    # (policy, annotator) -> rows of that policy, 0/1 per category
for (pol, a), g in ann.groupby(["policy_file", "annotator"]):
    m = np.zeros((len(te), len(CATS)), int)
    for s, c in zip(g.segment_id, g.category):
        m[row[(pol, s)], cidx[c]] = 1
    mats[(pol, a)] = m
print("annotators per test policy:", pd.Series([p for p, a in mats]).value_counts().unique())

pair_true, pair_pred, bert_true, bert_pred = [], [], [], []
pred_bert = (p_bert > 0.5).astype(int)
for pol in pols:
    mask = (te.policy_file == pol).values
    anns = [m[mask] for (p, a), m in mats.items() if p == pol]
    for i, a in enumerate(anns):
        bert_true.append(a); bert_pred.append(pred_bert[mask])
        for j, b in enumerate(anns):
            if i != j:
                pair_true.append(b); pair_pred.append(a)

ceiling = prf(np.vstack(pair_true), np.vstack(pair_pred), zero_division=0)[2]
model = prf(np.vstack(bert_true), np.vstack(bert_pred), zero_division=0)[2]
out = pd.DataFrame({"annotator vs annotator": ceiling, "BERT vs annotator": model}, index=CATS)
out.loc["macro avg"] = out.mean()
out.round(3)

annotators per test policy: [3]


,annotator vs annotator,BERT vs annotator
First Party Collection/Use,0.841,0.833
Third Party Sharing/Collection,0.840,0.814
User Choice/Control,0.680,0.664
"User Access, Edit and Deletion",0.740,0.772
Data Retention,0.488,0.484
Data Security,0.696,0.786
Policy Change,0.747,0.752
Do Not Track,0.875,0.857
International and Specific Audiences,0.927,0.864
Other,0.660,0.658


**How to read it.** If BERT vs annotator is at or above annotator vs annotator, the labels are the limit, not the model. Then stop testing encoders and write that as the finding. If BERT is clearly below, there is room left.

## Part 2a. Granularity: paragraphs vs sentences
The classifier is trained on paragraphs (about 55 words) but the Gap Analyser sends it single sentences (about 17 words). Here we cut the test paragraphs into sentences, give each sentence the labels of its paragraph, and score.

In [13]:
sent = []
for i, t in enumerate(te.text):
    for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"(])", t):
        if len(s.split()) >= 4:
            sent.append((i, s))
si = np.array([i for i, _ in sent]); stext = [s for _, s in sent]
ys = y[si]
print("paragraphs:", len(te), "| sentences:", len(stext),
      "| words per paragraph:", round(te.text.str.split().str.len().mean(), 1),
      "| words per sentence:", round(np.mean([len(s.split()) for s in stext]), 1))

ps_base = base["clf"].predict_proba(base["vec"].transform(stext))
res = {"baseline paragraphs": prf(y, (p_base > .5).astype(int), zero_division=0)[2],
       "baseline sentences": prf(ys, (ps_base > .5).astype(int), zero_division=0)[2]}

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained("dralsarrani/opp115-bert")
mdl = AutoModelForSequenceClassification.from_pretrained("dralsarrani/opp115-bert").eval()
out = []
for i in range(0, len(stext), 32):
    enc = tok(stext[i:i+32], truncation=True, max_length=256, padding=True, return_tensors="pt")
    with torch.no_grad():
        out.append(torch.sigmoid(mdl(**enc).logits).numpy())
ps_bert = np.vstack(out)
res["BERT paragraphs"] = prf(y, (p_bert > .5).astype(int), zero_division=0)[2]
res["BERT sentences"] = prf(ys, (ps_bert > .5).astype(int), zero_division=0)[2]

g = pd.DataFrame(res, index=CATS)
g.loc["macro avg"] = g.mean()
g.round(3)

paragraphs: 663 | sentences: 1691 | words per paragraph: 66.7 | words per sentence: 26.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,baseline paragraphs,baseline sentences,BERT paragraphs,BERT sentences
First Party Collection/Use,0.810,0.694,0.865,0.712
Third Party Sharing/Collection,0.736,0.600,0.833,0.600
User Choice/Control,0.693,0.511,0.685,0.503
"User Access, Edit and Deletion",0.741,0.515,0.808,0.534
Data Retention,0.462,0.326,0.588,0.298
Data Security,0.795,0.641,0.886,0.745
Policy Change,0.889,0.667,0.776,0.758
Do Not Track,0.800,0.696,0.800,0.696
International and Specific Audiences,0.832,0.662,0.869,0.730
Other,0.667,0.573,0.718,0.612


The drop from "paragraphs" to "sentences" is the cost of granularity. Part of it is unavoidable: one sentence rarely carries all categories of its paragraph, so recall falls by construction. Two fixes: classify the **paragraph the evidence sentence came from**, or train on sentences that inherit their paragraph's labels.

## Part 2b. Thresholds chosen by cross-validation, by policy (baseline)
Earlier we tuned thresholds on one validation split of 17 policies and it hurt. Here every threshold comes from out-of-fold predictions over train + validation (5 folds, whole policies per fold). Any category with fewer than 20 positives stays at 0.5. To isolate the threshold effect, the final model is trained once on train + validation and scored on test twice: at 0.5 and with the new thresholds.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GroupKFold

tv = df[df.policy_file.isin(split["train"] + split["val"])].reset_index(drop=True)
ytv = tv[CATS].values

def make():
    return (TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
            OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)))

oof = np.zeros(ytv.shape)
for tr, va in GroupKFold(n_splits=5).split(tv, groups=tv.policy_file):
    v, c = make()
    c.fit(v.fit_transform(tv.text[tr]), ytv[tr])
    oof[va] = c.predict_proba(v.transform(tv.text[va]))

grid = np.arange(0.1, 0.91, 0.05)
thr = np.full(len(CATS), 0.5)
for k in range(len(CATS)):
    if ytv[:, k].sum() >= 20:
        f = [prf(ytv[:, k], (oof[:, k] > t).astype(int), zero_division=0)[2][1] for t in grid]
        thr[k] = grid[int(np.argmax(f))]
print(dict(zip(CATS, thr.round(2))))

v, c = make()
c.fit(v.fit_transform(tv.text), ytv)
pt = c.predict_proba(v.transform(te.text))
a = prf(y, (pt > 0.5).astype(int), zero_division=0)[2]
b = prf(y, (pt > thr).astype(int), zero_division=0)[2]
print("baseline (train+val) test macro-F1 at 0.5:", a.mean().round(3), "| with CV thresholds:", b.mean().round(3))

{'First Party Collection/Use': np.float64(0.5), 'Third Party Sharing/Collection': np.float64(0.5), 'User Choice/Control': np.float64(0.55), 'User Access, Edit and Deletion': np.float64(0.55), 'Data Retention': np.float64(0.4), 'Data Security': np.float64(0.6), 'Policy Change': np.float64(0.65), 'Do Not Track': np.float64(0.25), 'International and Specific Audiences': np.float64(0.55), 'Other': np.float64(0.5)}
baseline (train+val) test macro-F1 at 0.5: 0.768 | with CV thresholds: 0.768


**Remember:** a change below the interval width from Part 1a is not evidence. Report it as "no clear gain" if it is small.